# Interpolation figures: gender and age

Reproduces the paper's two interpolation figures for the MLGD-F scribble of each continuum
scenario: **the sorted grid** (16 of the scribble's portraits ordered along the target axis)
and **the PCA scatter** (the target groups and the scribble's portraits). Nothing here is a
metric; the numbers are `four_targets/run.sh eval`.

For each scenario: a target is generated **on the scribble** (the four gender groups, or one
group per age 40–79), 100 portraits of the scribble are generated under the neutral prompt,
every portrait gets a score along the target axis (gender: PC1 of a PCA fitted on Woman vs
Man; age: the axis from the mean age-40 to the mean age-79 embedding), and the figures are
drawn from those scores. The heavy code is in `cdm/models/sd/interpolation.py`.

Run top to bottom on a GPU (L40S, L4 or A100; a 16 GB T4 runs out of memory while loading),
on the cluster or on Colab (first cell). The scribbles are the paper runs' in
`four_targets/results/`, so nothing needs uploading.

### How the portraits in the sorted grid are picked

The grid shows 16 of the 100 portraits, ordered by their score on the CLIP axis. That order is
only a proxy for what a person sees: two portraits a few ranks apart can look the other way
round, and lighting, hair or expression can move a portrait's CLIP score without changing its
perceived gender or age. So the picks are not exactly evenly spaced. We start from even
spacing (about every 6–7 ranks over the 100 portraits) and, within a few ranks of each
position, take the portrait whose perceived gender or age best shows that position.

The ranks used are fixed in `PICKS` below and saved to `picks.json`; `ranked_all.png` shows
every portrait with its rank, so the choice can be checked; `PICKS = None` gives the purely
evenly spaced grid. The ranks refer to these exact portraits: they hold for the same seed and
`N_EVAL` on the same GPU type and library versions (the run is deterministic).

In [ ]:
# --- Colab only: clone the repository (skipped on the cluster) ---------------------------
import os, sys, subprocess
if "google.colab" in sys.modules:
    from google.colab import userdata
    # a GitHub token with read access, saved as a Colab secret named GITHUB (key icon, left)
    if not os.path.exists("/content/CDM"):
        tok = userdata.get("GITHUB")
        subprocess.run(["git", "clone", "-q", "--branch", "master",
                        f"https://{tok}@github.com/orineo1/CDM.git", "/content/CDM"], check=True)
        subprocess.run(["git", "-C", "/content/CDM", "remote", "set-url", "origin",
                        "https://github.com/orineo1/CDM.git"], check=True)
    os.chdir("/content/CDM")
    subprocess.run("pip install -q -r requirements.txt -r experiments/stable_diffusion/requirements.txt",
                   shell=True, check=True)

In [ ]:
# --- Settings: the paper's figures --------------------------------------------------------
N_EVAL    = 100        # portraits of the scribble that are ranked
N_TARGET  = 300        # gender target: 75 per group (Woman, WomanMasc, ManFem, Man)
N_PER_AGE = 3          # age target: per age, 40-79
AGE_MIN, AGE_MAX = 40, 79
SEED      = 42

SCENARIOS = {
    "gender": dict(run="full_GenderInterpolation_seed8",
                   picks=[0, 8, 14, 20, 26, 31, 40, 45, 50, 58, 63, 75, 80, 85, 89, 98],
                   n_more=200),     # 200 more portraits (seed SEED+1) in the scatter: 300 in all
    "age":    dict(run="full_AgeInterpolation_seed2",
                   picks=[0, 6, 13, 19, 26, 33, 42, 48, 53, 59, 65, 73, 80, 86, 91, 95],
                   n_more=0),       # the scatter shows the 100 ranked portraits
}

In [ ]:
import os, sys
from pathlib import Path
import numpy as np, torch
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from PIL import Image

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "cdm").is_dir())
os.chdir(REPO); sys.path.insert(0, str(REPO))
from cdm.models.sd import interpolation as itp
from cdm.models.sd.models import load_models, make_deterministic
from cdm.models.sd.clip_utils import load_clip_model

SD      = REPO / "experiments/stable_diffusion"
RESULTS = SD / "four_targets/results"
OUT     = SD / "interpolation/outputs"
# colour scales that read without a colour bar
CMAP = {"gender": LinearSegmentedColormap.from_list("woman_man", ["#d6336c", "#c9c3d6", "#1971c2"]),
        "age":    LinearSegmentedColormap.from_list("young_old", ["#cfe3f5", "#08306b"])}

def pinned_revisions():   # four_targets/config/models.env -> {"architect_revision": sha, ...}
    out = {}
    for line in (SD / "four_targets/config/models.env").read_text().splitlines():
        if "=" in line and not line.startswith("#"):
            k, v = line.split("=", 1)
            out[k.strip().lower()] = v.split("#")[0].strip()
    return out

make_deterministic(SEED)
device = "cuda"
rev = pinned_revisions()
architect, sprinter = load_models(device, architect_revision=rev.get("architect_revision"),
                                  sprinter_revision=rev.get("sprinter_revision"),
                                  controlnet_revision=rev.get("controlnet_revision"))
del architect; torch.cuda.empty_cache()
clip_model, clip_processor = load_clip_model(device, revision=rev.get("clip_revision"))

In [ ]:
def run_scenario(name, run, picks, n_more):
    """Target, portraits, axis and the two figures of one scenario, saved to OUT/<run>/."""
    run_dir, out = RESULTS / run, OUT / run
    out.mkdir(parents=True, exist_ok=True)
    scribble_path = run_dir / "final_scribble_mlgd_f.png"
    scribble = Image.open(scribble_path).convert("RGB")

    shown = [("source portrait", run_dir / "source_portrait.png"),
             ("source scribble", run_dir / "scribble_source.png"), ("MLGD-F scribble", scribble_path)]
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    for ax, (t, p) in zip(axes, shown):
        ax.imshow(Image.open(p)); ax.set_title(t); ax.axis("off")
    plt.show()

    # the target, generated on the scribble
    if name == "gender":
        groups = itp.GENDER_GROUPS
        target = itp.build_target(sprinter, clip_model, clip_processor, scribble, groups,
                                  N_TARGET // len(groups), SEED, device)
    else:
        groups = [(str(a), itp.age_prompt(a)) for a in range(AGE_MIN, AGE_MAX + 1)]
        target = itp.build_target(sprinter, clip_model, clip_processor, scribble, groups,
                                  N_PER_AGE, SEED, device, seed_step=7)

    # the scribble's portraits and their position on the axis
    photos, embs = itp.eval_photos(sprinter, clip_model, clip_processor, scribble, N_EVAL, SEED, device)
    axis = itp.gender_axis(target) if name == "gender" else itp.age_axis(target, AGE_MIN, AGE_MAX)
    scores, _ = axis.project(embs)

    # every portrait with its rank: the sheet the picks are checked against
    itp.ranked_strip(photos, scores, save_path=out / "ranked_all.png"); plt.show()

    # the sorted grid
    fig, picks, idx = itp.sorted_grid(photos, scores, picks=picks, show_scores=False)
    fig.savefig(out / "sorted_grid.png", dpi=300, bbox_inches="tight"); plt.show()
    print("ranks shown:", picks)

    # the PCA scatter
    embs_all = embs
    if n_more:
        _, more = itp.eval_photos(sprinter, clip_model, clip_processor, scribble, n_more, SEED + 1, device)
        embs_all = np.concatenate([embs, more])
    color_by = {g: int(g) for g in target} if name == "age" else None
    fig = itp.scatter(axis, target, embs_all, color_by=color_by, cmap=CMAP[name])
    fig.axes[0].get_legend().remove()
    fig.savefig(out / "pca_scatter.png", dpi=300, bbox_inches="tight"); plt.show()

    itp.save_run(out, run, str(scribble_path), picks, idx, scores, embs, target,
                 extra={"seed": SEED, "n_eval": N_EVAL, "n_scatter": len(embs_all), "axis": axis.name})
    print(f"{name}: saved to {out}: {sorted(os.listdir(out))}")

## Gender interpolation

In [ ]:
run_scenario("gender", **SCENARIOS["gender"])

## Age interpolation

In [ ]:
run_scenario("age", **SCENARIOS["age"])

## Download (Colab)
The figures are in `experiments/stable_diffusion/interpolation/outputs/<run>/`:
`sorted_grid.png`, `pca_scatter.png`, `ranked_all.png`, `picks.json`, `embeddings.npz`.

In [ ]:
if "google.colab" in sys.modules:
    from google.colab import files
    import shutil
    shutil.make_archive("/content/interpolation_figures", "zip", OUT)
    files.download("/content/interpolation_figures.zip")